# Argus — Deployment Export (LSTM)

**Final stage of the LSTM pipeline.** Assembles the end-to-end deployment artifact
(`LstmGeometricFeatureModel`: raw MediaPipe outputs in, a 3-class softmax drowsiness class out),
saves it, reloads it to verify it deserializes correctly, and sanity-checks it against a real
video via a live-stream style simulation.

**Reads:** the latest trained LSTM (`models/lstm_drowsiness_*.keras`) and scaler
(`models/feature_scaler_*.joblib`) written by
[`03_model_training_lstm.ipynb`](./03_model_training_lstm.ipynb) — run that first. Also reads raw
videos from `dataset/raw_videos/` for the end-to-end simulation at the bottom.

**Writes:** `models/lstm_geometric_feature_model_<VERSION>.keras` — this is the artifact
`cv-argus` (`src/cv-argus/src/model/downloader.py`, via `MODEL_DRIVE_FILE_ID`) fetches for the
Raspberry Pi.

This notebook only covers the LSTM's deployment path. RandomForest, the Dense NN, and the
face-crop CNN (`04_random_forest_training.ipynb`, `05_dense_nn_training.ipynb`,
`07_cnn_training.ipynb`) don't have deployment-export notebooks of their own yet — the LSTM is
still the model the edge pipeline is being designed around (see
`03_model_training_lstm.ipynb`'s Design Decision cell).

⚠️ **`GeometricRatioFeatureLayer` is redefined below, verbatim.** It has to stay byte-identical
across four places: `01_dataset_creation_lstm.ipynb` (source of truth), `02_dataset_creation_flat.ipynb`,
this notebook, and `src/cv-argus/src/model/layers.py` (the Pi-side port — see that module's
docstring for why: Keras can't reconstruct a custom `call()` body from a saved file alone, so the
exact same class must be importable wherever the model is loaded). If you change this layer,
update all four and re-verify parity — there's no automated check for this yet.


## Setup

In [ ]:
from google.colab import drive
import os
import re
import glob

drive.mount('/content/drive')
project_folder = "/content/drive/MyDrive/Argus"
print(f"Google Drive successfully mounted! Base project directory: {project_folder}")


In [ ]:
models_folder = f"{project_folder}/models"
dataset_folder = f"{project_folder}/dataset"
raw_videos_folder = f"{dataset_folder}/raw_videos"

video_files = glob.glob(os.path.join(raw_videos_folder, "**/*.mp4"), recursive=True)
if len(video_files) == 0:
    raise Exception(f"No video clips found in '{raw_videos_folder}' — needed for the end-to-end simulation below.")
print(f"Found {len(video_files)} raw clip(s).")

# --- Feature/pipeline constants (must match 01_dataset_creation_lstm.ipynb) ---
sampling_fps = 10
pose_validity_threshold_deg = 20.0
MAX_TIMESTEPS = 60  # fixed pad length every training window was padded to -- see
                     # 01_dataset_creation_lstm.ipynb's "Pipeline Configuration Constants" cell.
                     # Brought down from 120 after a full extraction run at that size OOM-crashed
                     # the Colab kernel (peak RAM there scaled with MAX_TIMESTEPS * num_features
                     # per flattened window row, accumulated for every window before being
                     # written out). The deployed feature_buffer below must hold exactly this
                     # many frames, since that's the input shape the trained LSTM expects.
max_timesteps = MAX_TIMESTEPS  # kept for backward-compatible naming with the cells below


In [ ]:
import urllib.request
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

media_pipe_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
media_pipe_path = os.path.join(models_folder, "face_landmarker.task")

if not os.path.exists(media_pipe_path):
    print(f"Downloading MediaPipe Face Landmarker model to {media_pipe_path}...")
    urllib.request.urlretrieve(media_pipe_url, media_pipe_path)
else:
    print(f"Model already exists at {media_pipe_path}. Skipping download.")

base_options = mp_python.BaseOptions(model_asset_path=os.path.abspath(media_pipe_path))
face_landmarker_options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
)
print("FaceLandmarker configuration ready.")


### `GeometricRatioFeatureLayer` (verbatim port — see warning above)

In [ ]:
import tensorflow as tf
import numpy as np

class GeometricRatioFeatureLayer(tf.keras.layers.Layer):
    """
    TensorFlow-native counterpart to compute_ear, compute_mar, and rotation_matrix_to_euler.
    Inputs:
        landmarks_xy : (batch, 478, 2)   -- normalized (x, y) coordinates
        rotation_matrix : (batch, 3, 3)  -- top‑left 3×3 block of the facial transformation matrix
    Returns:
        (batch, 7) : [EAR_left, EAR_right, MAR, pitch, yaw, roll, ear_mar_valid]
        where pitch, yaw, roll are in degrees, and ear_mar_valid is 1 if the head pose is within
        the validity threshold, else 0.
    """
    def __init__(self, pose_validity_threshold_deg=20.0, **kwargs):
        super().__init__(**kwargs)
        self.pose_validity_threshold_deg = pose_validity_threshold_deg

        # Fixed landmark indices (same as used in the NumPy functions)
        self.left_eye_idx  = tf.constant([33, 160, 158, 133, 153, 144], dtype=tf.int32)
        self.right_eye_idx = tf.constant([362, 385, 387, 263, 373, 380], dtype=tf.int32)
        self.mouth_idx     = tf.constant([61, 291, 13, 14], dtype=tf.int32)

    def get_config(self):
        config = super().get_config()
        config.update({
            "pose_validity_threshold_deg": self.pose_validity_threshold_deg,
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

    @staticmethod
    def _dist(points, i, j):
        """Euclidean distance between two selected points, batch‑safe."""
        a = points[..., i, :]
        b = points[..., j, :]
        return tf.norm(a - b, axis=-1)

    def _ear(self, landmarks, idx):
        """
        Eye Aspect Ratio.
        idx convention: [left_corner, top_1, top_2, right_corner, bottom_2, bottom_1]
        """
        p = tf.gather(landmarks, idx, axis=-2)          # shape: (..., 6, 2)
        vertical = self._dist(p, 1, 5) + self._dist(p, 2, 4)
        horizontal = 2.0 * self._dist(p, 0, 3)
        return tf.math.divide_no_nan(vertical, horizontal)

    def _mar(self, landmarks, idx):
        """
        Mouth Aspect Ratio.
        idx convention: [left_corner, right_corner, upper_lip, lower_lip]
        """
        p = tf.gather(landmarks, idx, axis=-2)          # shape: (..., 4, 2)
        vertical = self._dist(p, 2, 3)
        horizontal = self._dist(p, 0, 1)
        return tf.math.divide_no_nan(vertical, horizontal)

    @staticmethod
    def _rotation_matrix_to_euler(R):
        """
        Convert a 3×3 rotation matrix to pitch, yaw, roll in degrees.
        Handles the gimbal‑lock singularity gracefully.
        """
        r00, r10, r20 = R[..., 0, 0], R[..., 1, 0], R[..., 2, 0]
        r21, r22 = R[..., 2, 1], R[..., 2, 2]
        r11, r12 = R[..., 1, 1], R[..., 1, 2]

        sy = tf.sqrt(r00**2 + r10**2)
        singular = sy < 1e-6

        pitch_reg = tf.atan2(r21, r22)
        yaw       = tf.atan2(-r20, sy)
        roll_reg  = tf.atan2(r10, r00)

        pitch_sing = tf.atan2(-r12, r11)
        roll_sing  = tf.zeros_like(roll_reg)

        pitch = tf.where(singular, pitch_sing, pitch_reg)
        roll  = tf.where(singular, roll_sing, roll_reg)

        rad2deg = 180.0 / np.pi
        return pitch * rad2deg, yaw * rad2deg, roll * rad2deg

    def call(self, landmarks_xy, rotation_matrix):
        ear_left  = self._ear(landmarks_xy, self.left_eye_idx)
        ear_right = self._ear(landmarks_xy, self.right_eye_idx)
        mar       = self._mar(landmarks_xy, self.mouth_idx)

        pitch, yaw, roll = self._rotation_matrix_to_euler(rotation_matrix)

        ear_mar_valid = tf.cast(
            tf.logical_and(
                tf.abs(yaw) < self.pose_validity_threshold_deg,
                tf.abs(pitch) < self.pose_validity_threshold_deg
            ),
            tf.float32
        )

        return tf.stack([ear_left, ear_right, mar, pitch, yaw, roll, ear_mar_valid], axis=-1)

print("✅ GeometricRatioFeatureLayer (TensorFlow-native geometric ratios) defined successfully.")

In [ ]:
# --- Ensure the ratio layer has the blendshape names (if not already defined) ---
if not hasattr(GeometricRatioFeatureLayer, 'blendshape_names'):
    GeometricRatioFeatureLayer.blendshape_names = [
        "browDownLeft", "browDownRight", "browInnerUp", "browOuterUpLeft", "browOuterUpRight",
        "cheekPuff", "cheekSquintLeft", "cheekSquintRight",
        "eyeBlinkLeft", "eyeBlinkRight",
        "eyeLookDownLeft", "eyeLookDownRight", "eyeLookInLeft", "eyeLookInRight",
        "eyeLookOutLeft", "eyeLookOutRight", "eyeLookUpLeft", "eyeLookUpRight",
        "eyeSquintLeft", "eyeSquintRight", "eyeWideLeft", "eyeWideRight",
        "jawForward", "jawLeft", "jawOpen", "jawRight",
        "mouthClose", "mouthDimpleLeft", "mouthDimpleRight", "mouthFrownLeft", "mouthFrownRight",
        "mouthFunnel", "mouthLeft", "mouthLowerDownLeft", "mouthLowerDownRight",
        "mouthPressLeft", "mouthPressRight", "mouthPucker", "mouthRight",
        "mouthRollLower", "mouthRollUpper", "mouthShrugLower", "mouthShrugUpper",
        "mouthSmileLeft", "mouthSmileRight", "mouthStretchLeft", "mouthStretchRight",
        "mouthUpperUpLeft", "mouthUpperUpRight", "noseSneerLeft", "noseSneerRight",
    ]

In [ ]:
import joblib
import datetime
import tensorflow as tf

def get_latest_model(folder, prefix, extension):
    if not os.path.exists(folder):
        return None
    files = [f for f in os.listdir(folder) if f.startswith(prefix) and f.endswith(extension)]
    if not files:
        return None
    return os.path.join(folder, sorted(files)[-1])

VERSION_STR = datetime.datetime.now().strftime("%Y%m%d_%H%M")

latest_lstm_path = get_latest_model(models_folder, "lstm_drowsiness", ".keras")
latest_scaler_path = get_latest_model(models_folder, "feature_scaler", ".joblib")
if not latest_lstm_path or not latest_scaler_path:
    raise FileNotFoundError(
        "No trained LSTM model / scaler found. Run 03_model_training_lstm.ipynb first."
    )

print(f"📦 Loading LSTM model: {latest_lstm_path}")
model = tf.keras.models.load_model(latest_lstm_path)

print(f"📦 Loading scaler: {latest_scaler_path}")
scaler = joblib.load(latest_scaler_path)

num_features = 7 + len(GeometricRatioFeatureLayer.blendshape_names)
ratio_layer = GeometricRatioFeatureLayer(pose_validity_threshold_deg=pose_validity_threshold_deg)

print(f"✅ Ready. num_features={num_features}, max_timesteps={max_timesteps}, version={VERSION_STR}")


## Geometric rate feature layer + LSTM Model for Deployment

To create a deployable model that takes raw MediaPipe outputs and directly provides drowsiness predictions, we need to wrap our feature extraction, normalization, and LSTM prediction logic into a single Keras model. This ensures that the entire inference pipeline is self-contained and can be saved as a single `.keras` file for efficient deployment.

### Overview of the Geometric rate feature layer + LSTM Pipeline:
1.  **Input**: Raw MediaPipe outputs (facial landmarks, rotation matrix, blendshape scores).
2.  **Geometric Feature Extraction**: A custom `GeometricRatioFeatureLayer` computes EAR, MAR, and head pose from landmarks and the rotation matrix.
3.  **Feature Combination**: Geometric features are concatenated with raw blendshape scores.
4.  **Sequence Accumulation & Padding**: Single-frame features are collected and padded to a fixed `max_timesteps` length.
5.  **Normalization**: A `tf.keras.layers.Normalization` layer standardizes the features using statistics learned during training.
6.  **LSTM Prediction**: The normalized sequence is fed into the pre-trained LSTM model for drowsiness classification.
7.  **Output**: Drowsiness level prediction.

In [ ]:
import joblib
import tensorflow as tf

# Ensure the scaler object is loaded and num_features are defined
# (These should be available from previous cells, but we check for robustness)
if 'scaler' not in globals():
    latest_scaler_path = get_latest_model(models_folder, "feature_scaler", ".joblib")
    if latest_scaler_path:
        print(f"Loading scaler from {latest_scaler_path}")
        scaler = joblib.load(latest_scaler_path)
    else:
        raise FileNotFoundError("No scaler found. Please train the LSTM model first (03_model_training_lstm.ipynb).")

if 'num_features' not in globals():
    # Fallback if num_features wasn't explicitly defined earlier (should be 59)
    num_features = 7 + len(GeometricRatioFeatureLayer.blendshape_names)

# Create a Keras Normalization layer and set its mean and variance from the trained StandardScaler
norm_layer = tf.keras.layers.Normalization(axis=-1) # Normalize per feature

# Explicitly build the layer with the expected input shape
# The input to normalization_layer in LstmGeometricFeatureModel.call is `sequence_input`
# which has shape (batch, max_timesteps, features)
norm_layer.build(input_shape=(None, None, num_features))

norm_layer.set_weights([scaler.mean_, scaler.var_, tf.constant(1.0, dtype=tf.float32)])

print(f"✅ Keras Normalization layer initialized with {num_features} features.")

### LstmGeometricFeatureModel Class

This custom Keras `Model` subclass encapsulates the entire prediction pipeline from raw MediaPipe outputs to drowsiness levels. It integrates:

*   **Input**: Raw facial landmarks (`(batch, 478, 2)`), rotation matrix (`(batch, 3, 3)`), and blendshape scores (`(batch, 52)`).
*   **`GeometricRatioFeatureLayer`**: Computes EAR, MAR, Pitch, Yaw, Roll, and validity from landmarks and rotation.
*   **Concatenation**: Combines geometric features with blendshape scores.
*   **Sequence Accumulation & Padding**: This model is designed for real-time inference, where frames arrive one by one. It accumulates `max_timesteps` worth of single-frame features, padding the sequence if fewer than `max_timesteps` are available. This is crucial for feeding a fixed-size sequence to the LSTM.
*   **`Normalization` Layer**: Applies the pre-trained feature scaling.
*   **Pre-trained LSTM Model**: The core prediction model.

This structure ensures that the deployed model handles all necessary preprocessing internally.

In [ ]:
import tensorflow as tf

class LstmGeometricFeatureModel(tf.keras.Model):
    def __init__(self, ratio_layer: GeometricRatioFeatureLayer,
                 normalization_layer: tf.keras.layers.Normalization,
                 lstm_model: tf.keras.Model,
                 max_timesteps: int = 60,
                 num_features: int = 58, **kwargs):
        super().__init__(**kwargs)
        self.ratio_layer = ratio_layer
        self.normalization_layer = normalization_layer
        self.lstm_model = lstm_model
        self.max_timesteps = max_timesteps
        self.num_features = num_features

        # Initialize a buffer for the sequence history (for real-time inference)
        # Note: This buffer is stateful, so it's handled within the model for deployment.
        self.feature_buffer = tf.Variable(
            tf.zeros([1, self.max_timesteps, self.num_features], dtype=tf.float32),
            trainable=False, # This buffer is not part of model training
            name="feature_buffer"
        )

        # Ensure blendshape names are available to the ratio_layer for consistent ordering
        if not hasattr(self.ratio_layer, 'blendshape_names'):
            raise AttributeError("ratio_layer must have 'blendshape_names' attribute defined.")

    def call(self, inputs, training=False):
        # inputs is a dictionary: {'landmarks': (1, 478, 2), 'rotation_matrix': (1, 3, 3), 'blendshapes': (1, 52)}
        landmarks = inputs['landmarks']
        rotation_matrix = inputs['rotation_matrix']
        raw_blendshapes = inputs['blendshapes']

        # 1. Geometric Feature Extraction
        geometric_features = self.ratio_layer(landmarks, rotation_matrix)

        # 2. Combine Geometric Features and Blendshapes
        # Ensure raw_blendshapes matches the expected order if necessary, but here we assume it's ordered.
        combined_features = tf.concat([geometric_features, raw_blendshapes], axis=-1)

        # Reshape to (1, 1, num_features) to represent a single frame's features
        single_frame_features = tf.reshape(combined_features, [1, 1, self.num_features])

        # 3. Update the feature buffer (sliding window)
        # Shift old features to the left, insert new feature at the right
        updated_buffer = tf.concat(
            [self.feature_buffer[:, 1:, :], single_frame_features],
            axis=1
        )
        self.feature_buffer.assign(updated_buffer)

        # Use the current state of the buffer as the sequence input for the LSTM
        sequence_input = self.feature_buffer

        # 4. Normalization
        normalized_sequence = self.normalization_layer(sequence_input)

        # 5. LSTM Prediction
        predictions = self.lstm_model(normalized_sequence, training=training)

        return predictions

    def get_config(self):
        config = super().get_config()
        config.update({
            "ratio_layer": tf.keras.utils.serialize_keras_object(self.ratio_layer),
            "normalization_layer": tf.keras.utils.serialize_keras_object(self.normalization_layer),
            "lstm_model": tf.keras.utils.serialize_keras_object(self.lstm_model),
            "max_timesteps": self.max_timesteps,
            "num_features": self.num_features,
        })
        return config

    @classmethod
    def from_config(cls, config):
        # Deserialize nested Keras objects
        ratio_layer_config = config.pop("ratio_layer")
        normalization_layer_config = config.pop("normalization_layer")
        lstm_model_config = config.pop("lstm_model")

        # Provide custom_objects for custom layers during deserialization
        ratio_layer = tf.keras.utils.deserialize_keras_object(ratio_layer_config, custom_objects={'GeometricRatioFeatureLayer': GeometricRatioFeatureLayer})
        normalization_layer = tf.keras.utils.deserialize_keras_object(normalization_layer_config)
        lstm_model = tf.keras.utils.deserialize_keras_object(lstm_model_config)

        # Instantiate the class with the deserialized objects and remaining config
        return cls(ratio_layer=ratio_layer,
                   normalization_layer=normalization_layer,
                   lstm_model=lstm_model,
                   **config)

print("✅ LstmGeometricFeatureModel class defined.")

### Instantiating and Building the Geometric rate feature layer + LSTM Model

Now we instantiate our `LstmGeometricFeatureModel` by providing it with the `GeometricRatioFeatureLayer`, the Keras `Normalization` layer, and the trained LSTM model. Crucially, we build the model by calling it with dummy inputs. This step is necessary for Keras `Model` subclasses to trace and create their internal graph, making the model ready for summary display and saving.

We also ensure all necessary components (`model`, `ratio_layer`, `scaler`, `max_timesteps`, `num_features`, `pose_validity_threshold_deg`) are available in the environment from previous cells.

In [ ]:
import joblib
import tensorflow as tf

# Ensure required components are loaded/defined
if 'model' not in globals():
    latest_lstm_path = get_latest_model(models_folder, "lstm_drowsiness", ".keras")
    if latest_lstm_path:
        print(f"Loading LSTM model from {latest_lstm_path}")
        model = tf.keras.models.load_model(latest_lstm_path)
    else:
        raise FileNotFoundError("No LSTM model found. Please train the LSTM model first.")

if 'ratio_layer' not in globals():
    if 'pose_validity_threshold_deg' not in globals():
        pose_validity_threshold_deg = 20.0 # Default value if not defined
        print(f"Using default pose_validity_threshold_deg: {pose_validity_threshold_deg}")
    ratio_layer = GeometricRatioFeatureLayer(pose_validity_threshold_deg=pose_validity_threshold_deg)

if 'norm_layer' not in globals():
    # This should have been created in a preceding cell, but as a fallback:
    if 'scaler' not in globals():
        latest_scaler_path = get_latest_model(models_folder, "feature_scaler", ".joblib")
        if latest_scaler_path:
            scaler = joblib.load(latest_scaler_path)
        else:
            raise FileNotFoundError("Scaler not found. Ensure it's loaded or created.")
    if 'num_features' not in globals():
        num_features = 7 + len(GeometricRatioFeatureLayer.blendshape_names)
    norm_layer = tf.keras.layers.Normalization(axis=-1)
    norm_layer.set_weights([scaler.mean_, scaler.var_])

# Instantiate the LstmGeometricFeatureModel
lstm_geometric_feature_model = LstmGeometricFeatureModel(
    ratio_layer=ratio_layer,
    normalization_layer=norm_layer,
    lstm_model=model,
    max_timesteps=max_timesteps,
    num_features=num_features
)

# Build the model by calling it with dummy inputs
# The input to the LstmGeometricFeatureModel call method is a dictionary of Tensors.
# These shapes should be for a single frame (batch size 1).
example_landmarks = tf.zeros([1, 478, 2], dtype=tf.float32)
example_rotation_matrix = tf.eye(3, batch_shape=[1], dtype=tf.float32)
example_blendshapes = tf.zeros([1, len(GeometricRatioFeatureLayer.blendshape_names)], dtype=tf.float32)

dummy_inputs = {
    'landmarks': example_landmarks,
    'rotation_matrix': example_rotation_matrix,
    'blendshapes': example_blendshapes
}

# Call the model with dummy inputs to build its graph
_ = lstm_geometric_feature_model(dummy_inputs)

print("\n✅ Geometric rate feature layer + LSTM Keras Model instantiated and built.")
lstm_geometric_feature_model.summary()

### Saving the Geometric rate feature layer + LSTM Model

To ensure our complete drowsiness detection system can be deployed easily, we save the `lstm_geometric_feature_model` in the Keras native format. This format preserves the entire model architecture, including custom layers, learned weights, the normalization parameters, and the internal state (`feature_buffer`), allowing for seamless loading and inference in a production environment.

The model will be saved to our `models_folder` with a version string for tracking.

In [ ]:
import os
import tensorflow as tf

# Ensure VERSION_STR and models_folder are available from previous cells
if 'VERSION_STR' not in globals():
    import datetime
    VERSION_STR = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    print(f"Generated VERSION_STR: {VERSION_STR}")

if 'models_folder' not in globals():
    raise NameError("models_folder is not defined. Ensure project setup cells are run.")

# Define the path for the end-to-end model with the new name
end_to_end_model_path = os.path.join(models_folder, f"lstm_geometric_feature_model_{VERSION_STR}.keras")

# Save the end-to-end model
lstm_geometric_feature_model.save(end_to_end_model_path)

print(f"✅ Geometric rate feature layer + LSTM Model saved successfully to: {end_to_end_model_path}")

In [ ]:
import tensorflow as tf
import os

# Ensure `end_to_end_model_path` is correctly defined (it should be from the saving cell)
if 'end_to_end_model_path' not in globals():
    # Fallback to reconstruct the path if the variable is not in scope
    if 'models_folder' in globals() and 'VERSION_STR' in globals():
        end_to_end_model_path = os.path.join(models_folder, f"lstm_geometric_feature_model_{VERSION_STR}.keras")
    else:
        raise NameError("Could not determine `end_to_end_model_path`. Please ensure previous cells were run.")


# Load the saved model
# We need to provide custom_objects if LstmGeometricFeatureModel or GeometricRatioFeatureLayer are custom Keras classes
# This is done by passing a dictionary to the custom_objects argument of load_model
loaded_model = tf.keras.models.load_model(
    end_to_end_model_path,
    custom_objects={'LstmGeometricFeatureModel': LstmGeometricFeatureModel,
                    'GeometricRatioFeatureLayer': GeometricRatioFeatureLayer}
)

print(f"✅ Model loaded successfully from: {end_to_end_model_path}")
print(f"Type of loaded model: {type(loaded_model)}")

# Optionally, you can also print a summary of the loaded model to verify its structure
loaded_model.summary()

## End-to-End Live Stream Simulation

This simulation bridges the gap between raw video and inference. It runs the entire pipeline directly on raw video frames rather than any precomputed dataset artifact:
**Raw Video Frame → MediaPipe Detection → `LstmGeometricFeatureModel` → Prediction.**

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import random
import os
from mediapipe.tasks.python import vision
import mediapipe as mp

# 1. Select a random video from the original dataset
random_video_path = random.choice(video_files)
subject_id = os.path.basename(os.path.dirname(random_video_path))
filename = os.path.basename(random_video_path)

# Extract ground truth class from filename -- same subject-number-aware convention as
# 01_dataset_creation_lstm.ipynb's "Video Processing Loop": subject_<N> with N >= EXTERNAL_SUBJECT_START
# already uses the final 1-3 class directly; Argus's own subject_01..subject_06 uses the
# original 1-6 scale and needs the same mapping applied there.
ORIGINAL_LEVEL_TO_CLASS = {1: 1, 2: 1, 3: 2, 4: 2, 5: 3, 6: 3}
CLASS_NAMES = {1: "Alert", 2: "Low Vigilant", 3: "Drowsy"}
EXTERNAL_SUBJECT_START = 7

level_match = re.search(r'level_(\d+)', filename, re.IGNORECASE)
if level_match:
    parsed_level = int(level_match.group(1))
    subject_num_match = re.search(r'(\d+)', subject_id)
    subject_num = int(subject_num_match.group(1)) if subject_num_match else None
    if subject_num is not None and subject_num >= EXTERNAL_SUBJECT_START:
        actual_class = parsed_level
    else:
        actual_class = ORIGINAL_LEVEL_TO_CLASS.get(parsed_level)
else:
    actual_class = None

print(f"--- Live Simulation Starting ---")
print(f"Target Video: {filename} (Subject: {subject_id})")
print(f"Ground Truth Class: {CLASS_NAMES.get(actual_class, 'Unknown')}\n")

# 2. Setup Video Capture and Local Detector
cap = cv2.VideoCapture(random_video_path)
src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
frame_stride = max(1, round(src_fps / sampling_fps))

# Initialize a fresh detector for this video to reset MediaPipe's internal clock
local_detector = vision.FaceLandmarker.create_from_options(face_landmarker_options)

# 3. Stream frames at 10 FPS
frame_count = 0
processed_count = 0
max_sim_frames = max_timesteps  # Simulate a full buffer's worth of frames (max_timesteps / sampling_fps seconds)

print(f"Processing frames at {sampling_fps} FPS...")

while processed_count < max_sim_frames:
    ret, frame_bgr = cap.read()
    if not ret: break

    if frame_count % frame_stride == 0:
        # Prepare MediaPipe Image
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        timestamp_ms = int(frame_count * (1000.0 / src_fps))

        # Run MediaPipe
        result = local_detector.detect_for_video(mp_image, timestamp_ms)

        if result.face_landmarks:
            # Extract raw components needed by the model dictionary
            lm = result.face_landmarks[0]
            landmarks_xy = tf.constant([[p.x, p.y] for p in lm], dtype=tf.float32)[tf.newaxis, ...]

            R = np.array(result.facial_transformation_matrixes[0])[:3, :3] if result.facial_transformation_matrixes else np.eye(3, dtype=np.float32)
            R_tf = tf.constant(R, dtype=tf.float32)[tf.newaxis, ...]

            bs_dict = {b.category_name: b.score for b in result.face_blendshapes[0]} if result.face_blendshapes else {}
            bs_scores = tf.constant([[bs_dict.get(name, 0.0) for name in GeometricRatioFeatureLayer.blendshape_names]], dtype=tf.float32)

            # Create input dictionary
            model_inputs = {
                'landmarks': landmarks_xy,
                'rotation_matrix': R_tf,
                'blendshapes': bs_scores
            }

            # Feed into the assembled model
            # This updates the internal buffer and produces a prediction
            probs = loaded_model(model_inputs, training=False)
            processed_count += 1
        else:
            # Optional: handle 'face lost' in a real app (e.g., reset buffer)
            pass

    frame_count += 1

cap.release()
local_detector.close()

# 4. Final Assessment
final_probs = probs.numpy()
predicted_idx = np.argmax(final_probs, axis=1)[0]
predicted_class = predicted_idx + 1
confidence = final_probs[0][predicted_idx]

print(f"\n--- Simulation Results ---")
print(f"Final Predicted Class: {CLASS_NAMES.get(predicted_class, 'Unknown')}")
print(f"Actual Ground Truth: {CLASS_NAMES.get(actual_class, 'Unknown')}")
print(f"Confidence: {confidence:.2%}")

if predicted_class == actual_class:
    print("\u2705 SUCCESS: The model correctly identified the class from raw video stream!")
else:
    print("\u274c MISMATCH: The model output differs from the ground truth.")